In [1]:
from pathlib import Path

# Notebooks may start in the project folder or the notebooks folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Data folder not found: {DATA_DIR}")

files = sorted(DATA_DIR.glob("*.csv"))

print(f"Data folder: {DATA_DIR}")
print(f"CSV files found: {len(files)}\n")

for file in files:
    size_mb = file.stat().st_size / (1024 ** 2)
    print(f"{file.name}: {size_mb:,.1f} MB")

Data folder: /Users/mariutzadanilescu/Documents/Coding projects/kkbox-music-recommendation/data/raw
CSV files found: 6

members.csv: 2.4 MB
sample_submission.csv: 28.2 MB
song_extra_info.csv: 172.6 MB
songs.csv: 211.6 MB
test.csv: 331.7 MB
train.csv: 926.7 MB


In [2]:
import pandas as pd

# Read only five rows while we inspect the structure.
train_preview = pd.read_csv(DATA_DIR / "train.csv", nrows=5)

print("Training columns:")
print(train_preview.columns.tolist())

display(train_preview)

Training columns:
['msno', 'song_id', 'source_system_tab', 'source_screen_name', 'source_type', 'target']


,msno,song_id,source_system_tab,source_screen_name,source_type,target
0,FGtllVqz18RPiwJj/edr2gV78zirAiY/9SmYvia+kCg=,BBzumQNXUHKdEBOB7mAJuzok+IJA1c2Ryg/yzTF6tik=,explore,Explore,online-playlist,1
1,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,bhp/MpSNoqoxOIB+/l8WPqu6jldth4DIpCm3ayXnJqM=,my library,Local playlist more,local-playlist,1
2,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,JNWfrrC7zNN7BdMpsISKa4Mw+xVJYNnxXh3/Epw7QgY=,my library,Local playlist more,local-playlist,1
3,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,2A87tzfnJTSWqD7gIZHisolhe4DMdzkbd6LzO1KHjNs=,my library,Local playlist more,local-playlist,1
4,FGtllVqz18RPiwJj/edr2gV78zirAiY/9SmYvia+kCg=,3qm6XTZ6MOCU11x8FIVbAGH5l5uMkT3/ZalWG1oo2Gc=,explore,Explore,online-playlist,1


In [3]:
# Read only the target column, in manageable chunks.
target_counts = pd.Series(0, index=[0, 1], dtype="int64")

for chunk in pd.read_csv(
    DATA_DIR / "train.csv",
    usecols=["target"],
    dtype={"target": "int8"},
    chunksize=250_000,
):
    counts = chunk["target"].value_counts()
    target_counts = target_counts.add(counts, fill_value=0)

target_summary = pd.DataFrame({
    "count": target_counts.astype("int64"),
    "percentage": (target_counts / target_counts.sum() * 100).round(2),
})
target_summary.index.name = "target"

print(f"Total training rows: {int(target_counts.sum()):,}")
display(target_summary)

Total training rows: 7,377,418


,count,percentage
target,,
0,3662762,49.65
1,3714656,50.35
